# 05 — Few-Shot v1 1ex

**Amostra:** `data/sample/toldBr_sample_500.csv` (500 tweets estratificados)

1 exemplo por categoria, sem descrições, sem anti-bias.

- Saída: `results/few_shot_v1_1ex.csv`

In [ ]:
import time
from pathlib import Path
import polars as pl
import requests
from sklearn.metrics import classification_report, f1_score

OLLAMA_BASE = "http://127.0.0.1:11434"
MODEL = "qwen3.5:9b"
VALID_LABELS = {"NOT_TOXIC", "OBSCENE", "INSULT", "HOMOPHOBIA", "RACISM", "MISOGYNY", "XENOPHOBIA"}
OUTPUT = "../results/few_shot_v1_1ex.csv"

Path("../results").mkdir(exist_ok=True)
sample = pl.read_csv("../data/sample/toldBr_sample_500.csv")
print(f"Tweets: {len(sample)}")

In [ ]:
EXAMPLES = """
Comentário: "Bom dia a todos! Que dia lindo hoje."
Classificação: NOT_TOXIC

Comentário: "Vai se f*der, seu lixo imundo!"
Classificação: OBSCENE

Comentário: "Você é um completo idiota, não sabe nada."
Classificação: INSULT

Comentário: "Viado não merece respeito."
Classificação: HOMOPHOBIA

Comentário: "Negro é tudo igual, não presta."
Classificação: RACISM

Comentário: "Mulher não serve pra nada além de cozinhar."
Classificação: MISOGYNY

Comentário: "Nordestino é tudo ignorante e preguiçoso."
Classificação: XENOPHOBIA
"""

def build_prompt(tweet: str) -> str:
    return (
        "Você é um sistema de moderação de conteúdo em português brasileiro.\n"
        "Classifique o comentário em UMA das categorias:\n"
        "NOT_TOXIC, OBSCENE, INSULT, HOMOPHOBIA, RACISM, MISOGYNY, XENOPHOBIA\n"
        "Responda APENAS com o nome da categoria.\n\n"
        f"Exemplos:{EXAMPLES}\n"
        f"Comentário: {tweet}\n"
        "Classificação:"
    )

def parse_label(response: str) -> str:
    text = response.strip().upper().replace(" ", "_")
    for label in VALID_LABELS:
        if label in text:
            return label.lower()
    return "unknown"

In [ ]:
resultados = []
t0 = time.time()

for i, row in enumerate(sample.iter_rows(named=True)):
    payload = {"model": MODEL, "prompt": build_prompt(row["text"]), "stream": False, "think": False}
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    data = r.json()
    predicao = parse_label(data["response"])
    tps = data["eval_count"] / (data["eval_duration"] / 1e9)
    resultados.append({"text": row["text"], "label": row["label"], "predicao": predicao,
                       "resposta_raw": data["response"].strip(), "tokens_s": round(tps, 1)})
    if (i + 1) % 50 == 0:
        print(f"{i+1}/500 — {time.time()-t0:.0f}s")

df = pl.DataFrame(resultados)
df.write_csv(OUTPUT)
print(f"\nConcluído em {time.time()-t0:.0f}s | UNKNOWN: {(df['predicao']=='unknown').sum()}")

In [ ]:
y_true, y_pred = df["label"].to_list(), df["predicao"].to_list()
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"F1-macro: {f1:.4f}\n")
print(classification_report(y_true, y_pred, zero_division=0))